# 08_reporting — 統合レポート生成

全ノートブックの出力を統合して HTML / Markdown レポートを生成する。

## 読み込むファイル
| ファイル | ソース |
|---|---|
| `reports/data_validation_report.md` | 02_data_validation |
| `reports/feature_summary.csv` | 03_feature_engineering |
| `reports/feature_importance.csv` | 04_feature_analysis |
| `reports/training_history.json` | 05_model_training |
| `reports/prediction.csv` | 06_prediction |
| `reports/evaluation_report.md` | 07_evaluation |

出力:
- `reports/full_report.md`
- `reports/full_report.html`

In [1]:
import sys, json
from pathlib import Path
_NB_DIR = Path().resolve()
if _NB_DIR.name != 'notebooks': _NB_DIR = _NB_DIR.parent
if str(_NB_DIR) not in sys.path: sys.path.insert(0, str(_NB_DIR))
from utils.nb_config import *

import pandas as pd
import numpy as np
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

print(f"レポートディレクトリ: {REPORTS_DIR}")
print("\n生成済みファイル:")
for f in sorted(REPORTS_DIR.glob('*')):
    print(f"  {f.name}  ({f.stat().st_size:,} bytes)")

レポートディレクトリ: C:\Users\yuki2\Documents\ws\keiba-ai-pro\notebooks\reports

生成済みファイル:
  analysis_report_notion.md  (13,639 bytes)
  calibration.png  (33,743 bytes)
  cv_score.png  (31,112 bytes)
  data_validation_report.md  (493 bytes)
  distribution.png  (50,272 bytes)
  evaluation_report.md  (300 bytes)
  feature_analysis_ex.csv  (7,297 bytes)
  feature_importance.csv  (12,653 bytes)
  feature_importance.png  (72,023 bytes)
  feature_llm_report.md  (6,779 bytes)
  feature_pruning_report.csv  (169 bytes)
  feature_pruning_sim.png  (27,350 bytes)
  feature_pruning_sim_ex.csv  (182 bytes)
  feature_rank_report.csv  (8,335 bytes)
  feature_summary.csv  (8,545 bytes)
  full_report.html  (347,327 bytes)
  full_report.md  (4,081 bytes)
  gpu_benchmark.csv  (251 bytes)
  gpu_usage_log.csv  (303 bytes)
  high_correlation.csv  (3,351 bytes)
  leakage_features.csv  (19 bytes)
  missing_report.csv  (13,572 bytes)
  missing_report_chart.png  (47,950 bytes)
  nan_rate_after_prep.png  (114,066 bytes)
 

In [2]:
# ── 各セクションを読み込む ────────────────────────────────────
def _read_file(p: Path, default: str = '') -> str:
    return p.read_text(encoding='utf-8') if p.exists() else default

_val_md   = _read_file(REPORTS_DIR / 'data_validation_report.md')
_eval_md  = _read_file(REPORTS_DIR / 'evaluation_report.md')
_hist_raw = _read_file(REPORTS_DIR / 'training_history.json', '{}')
_hist     = json.loads(_hist_raw)

_feat_sum = pd.read_csv(REPORTS_DIR / 'feature_summary.csv')    if (REPORTS_DIR / 'feature_summary.csv').exists()    else pd.DataFrame()
_feat_imp = pd.read_csv(REPORTS_DIR / 'feature_importance.csv') if (REPORTS_DIR / 'feature_importance.csv').exists() else pd.DataFrame()
_pred_df  = pd.read_csv(REPORTS_DIR / 'prediction.csv')         if (REPORTS_DIR / 'prediction.csv').exists()         else pd.DataFrame()
_hc_df    = pd.read_csv(REPORTS_DIR / 'high_correlation.csv')   if (REPORTS_DIR / 'high_correlation.csv').exists()   else pd.DataFrame()

print(f"データ検証レポート: {len(_val_md)} chars")
print(f"評価レポート:       {len(_eval_md)} chars")
print(f"学習履歴:           {list(_hist.keys())}")
print(f"特徴量サマリ:       {len(_feat_sum)} 列")
print(f"重要度:             {len(_feat_imp)} 列")
print(f"予測:               {len(_pred_df):,} 件")

データ検証レポート: 331 chars
評価レポート:       217 chars
学習履歴:           ['trained_at', 'target', 'task', 'cv_folds', 'optuna_trials', 'cv_neg_rmse_mean', 'cv_neg_rmse_std', 'test_auc', 'test_logloss', 'best_params', 'n_features']
特徴量サマリ:       146 列
重要度:             146 列
予測:               36,344 件


In [3]:
# ── Markdown 統合レポート生成 ──────────────────────────────────
_now = datetime.now().strftime('%Y-%m-%d %H:%M:%S')

_sections = []
_sections.append(f"""# 競馬 AI 予測システム — 分析レポート

**生成日時**: {_now}

---
""")

# 1. データ概要
_sections.append("## 1. データ品質\n")
_sections.append(_val_md)
_sections.append("\n---\n")

# 2. 特徴量概要
_sections.append("## 2. 特徴量概要\n")
if not _feat_sum.empty:
    _null_high = _feat_sum[_feat_sum['null_rate'] > 50] if 'null_rate' in _feat_sum.columns else pd.DataFrame()
    _sections.append(f"- 総特徴量数: **{len(_feat_sum)}**\n")
    _sections.append(f"- カテゴリ特徴量数: **{_feat_sum['is_categorical'].sum() if 'is_categorical' in _feat_sum.columns else 'N/A'}**\n")
    _sections.append(f"- 欠損率 > 50% の特徴量: **{len(_null_high)}**\n")
_sections.append("\n---\n")

# 3. 特徴量重要度
_sections.append("## 3. 特徴量重要度 (LightGBM Gain Top20)\n")
if not _feat_imp.empty and 'lgb_gain' in _feat_imp.columns:
    _top20 = _feat_imp.nlargest(20, 'lgb_gain')[['feature','lgb_gain','spearman_r']]
    _sections.append(_top20.to_markdown(index=False))
    _sections.append("\n")
_sections.append("\n---\n")

# 4. 学習結果
_sections.append("## 4. モデル学習\n")
if _hist:
    _sections.append(f"- 学習日時:      {_hist.get('trained_at','N/A')}\n")
    _sections.append(f"- 目的変数:      {_hist.get('target','N/A')}\n")
    _sections.append(f"- CV AUC:        **{_hist.get('cv_auc_mean','N/A')} ± {_hist.get('cv_auc_std','N/A')}**\n")
    _sections.append(f"- Test AUC:      **{_hist.get('test_auc','N/A')}**\n")
    _sections.append(f"- Test LogLoss:  {_hist.get('test_logloss','N/A')}\n")
    _sections.append(f"- 特徴量数:      {_hist.get('n_features','N/A')}\n")
    _sections.append(f"- Optuna Trials: {_hist.get('optuna_trials','N/A')}\n")
_sections.append("\n---\n")

# 5. 高相関ペア
_sections.append("## 5. 高相関特徴量ペア (|r| ≥ 0.90)\n")
if not _hc_df.empty:
    _sections.append(f"- 該当ペア数: {len(_hc_df)}\n")
    _sections.append(_hc_df.head(10).to_markdown(index=False))
    _sections.append("\n")
else:
    _sections.append("- 高相関ペアなし\n")
_sections.append("\n---\n")

# 6. 評価
_sections.append("## 6. 評価\n")
_sections.append(_eval_md)
_sections.append("\n---\n")

# 7. 画像リスト
_img_list = ['distribution.png', 'feature_importance.png', 'shap_summary.png',
             'calibration.png', 'profit_simulation.png', 'cv_auc.png']
_existing_imgs = [i for i in _img_list if (REPORTS_DIR / i).exists()]
if _existing_imgs:
    _sections.append("## 7. 図一覧\n")
    for img in _existing_imgs:
        _sections.append(f"### {img}\n")
        _sections.append(f"![{img}]({img})\n\n")

_full_md = '\n'.join(_sections)
_md_path = REPORTS_DIR / 'full_report.md'
_md_path.write_text(_full_md, encoding='utf-8')
print(f"Markdown レポート保存: {_md_path} ({len(_full_md)} chars)")

Markdown レポート保存: C:\Users\yuki2\Documents\ws\keiba-ai-pro\notebooks\reports\full_report.md (3528 chars)


In [4]:
# ── HTML レポート生成 ────────────────────────────────────────
import base64

def _img_tag(img_name: str) -> str:
    """画像を base64 embed した img タグを返す"""
    p = REPORTS_DIR / img_name
    if not p.exists(): return ''
    _b64 = base64.b64encode(p.read_bytes()).decode()
    return f'<img src="data:image/png;base64,{_b64}" style="max-width:100%;margin:8px 0;"/>'

# Markdown → HTML（markdown ライブラリが利用可能な場合）
try:
    import markdown
    _html_body = markdown.markdown(
        _full_md,
        extensions=['tables', 'fenced_code']
    )
    _img_embed_used = False
except ImportError:
    # フォールバック: pre タグで wrap
    _html_body = f'<pre style="white-space:pre-wrap;">{_full_md}</pre>'
    _img_embed_used = False

# 画像埋め込みセクション
_img_section = '<section><h2>図一覧</h2>'
for img in _existing_imgs:
    _img_section += f'<h3>{img}</h3>' + _img_tag(img)
_img_section += '</section>'

_html = f"""<!DOCTYPE html>
<html lang="ja">
<head>
  <meta charset="utf-8">
  <meta name="viewport" content="width=device-width, initial-scale=1">
  <title>競馬AI予測システム 分析レポート</title>
  <style>
    body {{ font-family: 'Meiryo', 'Yu Gothic', sans-serif; max-width: 1200px;
            margin: 0 auto; padding: 16px; line-height: 1.6; }}
    table {{ border-collapse: collapse; width: 100%; margin: 8px 0; }}
    th, td {{ border: 1px solid #ddd; padding: 6px 12px; text-align: left; }}
    th {{ background: #f5f5f5; }}
    h1 {{ border-bottom: 2px solid #333; }}
    h2 {{ border-bottom: 1px solid #999; }}
    pre {{ background: #f8f8f8; padding: 12px; border-radius: 4px; overflow-x: auto; }}
    img {{ max-width: 100%; }}
  </style>
</head>
<body>
{_html_body}
{_img_section}
<footer><p>Generated: {_now}</p></footer>
</body>
</html>
"""

_html_path = REPORTS_DIR / 'full_report.html'
_html_path.write_text(_html, encoding='utf-8')
print(f"HTML レポート保存: {_html_path} ({len(_html):,} chars)")

HTML レポート保存: C:\Users\yuki2\Documents\ws\keiba-ai-pro\notebooks\reports\full_report.html (346,192 chars)


In [5]:
# ── サマリ表示 ─────────────────────────────────────────────────
print("="*60)
print("  競馬 AI 予測システム — 分析完了")
print("="*60)
print(f"  生成日時: {_now}")
print("")
print("  【学習結果】")
if _hist:
    print(f"    CV AUC:   {_hist.get('cv_auc_mean','N/A')} ± {_hist.get('cv_auc_std','N/A')}")
    print(f"    Test AUC: {_hist.get('test_auc','N/A')}")
print("")
print("  【出力ファイル】")
for f in sorted(REPORTS_DIR.glob('*')):
    print(f"    {f.name}")
print("="*60)

  競馬 AI 予測システム — 分析完了
  生成日時: 2026-07-04 18:13:42

  【学習結果】
    CV AUC:   N/A ± N/A
    Test AUC: nan

  【出力ファイル】
    analysis_report_notion.md
    calibration.png
    cv_score.png
    data_validation_report.md
    distribution.png
    evaluation_report.md
    feature_analysis_ex.csv
    feature_importance.csv
    feature_importance.png
    feature_llm_report.md
    feature_pruning_report.csv
    feature_pruning_sim.png
    feature_pruning_sim_ex.csv
    feature_rank_report.csv
    feature_summary.csv
    full_report.html
    full_report.md
    gpu_benchmark.csv
    gpu_usage_log.csv
    high_correlation.csv
    leakage_features.csv
    missing_report.csv
    missing_report_chart.png
    nan_rate_after_prep.png
    nan_rate_ranking.png
    optuna_trial_log.csv
    prediction.csv
    profit_simulation.png
    roi_cumulative.png
    roi_report.csv
    shap_summary.png
    system_usage_log.csv
    training_history.json


---
## Section 6 — LLM レポート & Notion 出力（feature_inspection より移行）

`feature_inspection.ipynb` Cells M1-M3 / P_Ollama / P_Notion より移行。

| セル | 内容 |
|---|---|
| P_Ollama | Ollama / OpenAI 接続確認 |
| M1 | FEATURE_STORE / REPORT_STORE から `feature_analysis_json` を構築 |
| M2 | LLM プロンプト設定 |
| M3 | LLM 呼び出し + `feature_llm_report.md` 保存 |
| P_Notion | 条件付き Notion ページ Export |


In [6]:
## ── P_Ollama: LLM バックエンド確認 ──────────────────────────────────────
# feature_inspection.ipynb Cell P_Ollama より移行

import subprocess, shutil, os

# ── LLM バックエンド設定 ──────────────────────────────────────────────────
# 優先順位: 環境変数 → Ollama ローカル → OpenAI 互換エンドポイント
LLM_BACKEND  = os.getenv("LLM_BACKEND", "ollama")      # "ollama" | "openai"
LLM_MODEL    = os.getenv("LLM_MODEL",   "qwen3:14b")   # Ollama モデル名
LLM_API_BASE = os.getenv("LLM_API_BASE","http://localhost:11434/v1")
LLM_API_KEY  = os.getenv("LLM_API_KEY", "ollama")      # OpenAI 互換の場合に使用

LLM_ENABLED  = False   # 初期値; 以下の確認で True に更新される

# ── Ollama 起動確認 ──────────────────────────────────────────────────────
if LLM_BACKEND == "ollama":
    _ollama_path = shutil.which("ollama")
    if _ollama_path:
        try:
            _res = subprocess.run(
                ["ollama", "list"], capture_output=True, text=True, timeout=10
            )
            _models_available = [line.split()[0] for line in _res.stdout.strip().split("\n")[1:]
                                  if line.strip()]
            print(f"Ollama 起動中  models: {_models_available}")
            if LLM_MODEL in _models_available or any(LLM_MODEL in m for m in _models_available):
                LLM_ENABLED = True
                print(f"  ✓ モデル '{LLM_MODEL}' 利用可能")
            else:
                print(f"  ⚠ '{LLM_MODEL}' が見つかりません。利用可能: {_models_available}")
                print(f"     → ターミナルで `ollama pull {LLM_MODEL}` を実行してください")
        except Exception as e:
            print(f"  ⚠ Ollama 確認エラー: {e}")
    else:
        print("  ⚠ ollama コマンドが見つかりません (Ollama 未インストール)")

elif LLM_BACKEND == "openai":
    try:
        import openai
        _client_test = openai.OpenAI(api_key=LLM_API_KEY, base_url=LLM_API_BASE)
        _models_test = _client_test.models.list()
        print(f"OpenAI 互換エンドポイント 接続成功: {LLM_API_BASE}")
        LLM_ENABLED = True
    except Exception as e:
        print(f"  ⚠ OpenAI 互換接続エラー: {e}")

print(f"\nLLM_ENABLED  : {LLM_ENABLED}")
print(f"LLM_BACKEND  : {LLM_BACKEND}")
print(f"LLM_MODEL    : {LLM_MODEL}")


Ollama 起動中  models: ['qwen3:14b', 'qwen3:4b']
  ✓ モデル 'qwen3:14b' 利用可能

LLM_ENABLED  : True
LLM_BACKEND  : ollama
LLM_MODEL    : qwen3:14b


In [7]:
## ── M1: feature_analysis_json を FEATURE_STORE / REPORT_STORE から構築 ──
# feature_inspection.ipynb Cell M1 より移行

import json

# ── FEATURE_STORE から JSON を読み込み ─────────────────────────────────────
_fa_json_path = FEATURE_STORE / "feature_analysis.json"
if _fa_json_path.exists():
    with open(_fa_json_path, "r", encoding="utf-8") as _f:
        feature_analysis_json = json.load(_f)
    print(f"✓ feature_analysis.json ロード: {len(feature_analysis_json)} キー")
else:
    print("⚠ FEATURE_STORE に feature_analysis.json が見つかりません。")
    print("  04_feature_analysis.ipynb の OUT セルを先に実行してください。")
    feature_analysis_json = {}

# ── REPORT_STORE から roi_stats を読み込み ────────────────────────────────
_roi_stats = load_store(REPORT_STORE, "roi_stats")
if _roi_stats:
    feature_analysis_json["roi_stats"] = _roi_stats
    print(f"✓ roi_stats 統合: {_roi_stats}")

# ── model metrics を MODEL_STORE から追加 ─────────────────────────────────
_bundle_rp = load_store(MODEL_STORE, f"lgb_model_{TARGET}")
if _bundle_rp:
    feature_analysis_json["model_metrics"]  = _bundle_rp.get("metrics", {})
    feature_analysis_json["model_target"]   = _bundle_rp.get("target", TARGET)
    feature_analysis_json["best_iteration"] = _bundle_rp.get("best_iteration")
    print(f"✓ model metrics 統合: {_bundle_rp.get('metrics', {})}")

print("\n── feature_analysis_json 完成 ──")
for k in list(feature_analysis_json.keys())[:10]:
    _v = feature_analysis_json[k]
    if isinstance(_v, list): print(f"  {k}: [{len(_v)} items]")
    elif isinstance(_v, dict): print(f"  {k}: {_v}")
    else: print(f"  {k}: {_v}")


✓ feature_analysis.json ロード: 11 キー
  ✓ report_store/roi_stats.pkl ロード (0.2 KB)
✓ roi_stats 統合: {'top1_win_rate': np.float64(0.0932), 'top1_show_rate': np.float64(0.3354), 'top3_hit_rate': np.float64(0.3354), 'roi_top1_pct': np.float64(37.6), 'n_races': 161}
  ⚠ model_store/lgb_model_win.pkl が見つかりません

── feature_analysis_json 完成 ──
  target: speed_deviation
  created_at: 2026-06-28T21:54:47.185666
  n_features_total: 105
  n_features_ranked: 105
  top30_final_rank: [30 items]
  group_scores: [6 items]
  high_corr_pairs: [20 items]
  high_vif: [10 items]
  pruning_sim: [6 items]
  recommended_n_features: 105


In [8]:
## ── M2: LLM プロンプト設定 ────────────────────────────────────────────────
# feature_inspection.ipynb Cell M2 より移行

# ── プロンプト組み立て ──────────────────────────────────────────────────
_top30 = feature_analysis_json.get("top30_final_rank", [])
_groups = feature_analysis_json.get("group_scores", [])
_prune  = feature_analysis_json.get("pruning_sim", [])
_roi    = feature_analysis_json.get("roi_stats", {})
_metrics= feature_analysis_json.get("model_metrics", {})
_q_sum  = feature_analysis_json.get("quality_summary", {})
_n_rec  = feature_analysis_json.get("recommended_n_features", "N/A")
_high_corr = feature_analysis_json.get("high_corr_pairs", [])

_top30_text = "\n".join(
    f"  {r['rank']:>3}. {r['feature']:<45} score={r.get('final_score', 0):.4f}  group={r.get('group','')}"
    for r in _top30
)
_group_text = "\n".join(
    f"  {g['group']}: sum={g.get('sum_score',0):.3f}  avg={g.get('avg_score',0):.3f}  n={g.get('n_feats',0)}"
    for g in _groups
)
_prune_text = "\n".join(
    f"  n_feats={p.get('n_features',0)}: {p.get('cv_metric','score')}={p.get('cv_mean',0):.4f}"
    for p in _prune
)
_corr_text  = "\n".join(
    f"  {c.get('feature_a','')} ↔ {c.get('feature_b','')} : r={c.get('pearson_r',0):.3f}"
    for c in _high_corr[:10]
)

SYSTEM_PROMPT = """\
あなたは競馬AI・機械学習・LightGBM・SHAP分析・JRA競馬データ分析の専門家です。
必ず日本語で回答してください。英語での回答は禁止です。
与えられたデータに基づき、実用的かつ競馬理論と整合した分析レポートを作成してください。
単なる数値の羅列は禁止です。必ず「その特徴量が何を意味するか」「なぜ重要なのか」「競馬理論と整合するか」まで説明してください。
回答は Markdown 形式で、見出し・表・箇条書きを積極的に活用してください。
"""

USER_PROMPT = f"""/no_think
【重要】必ず日本語で回答してください。英語での回答は禁止です。

あなたは競馬AI・機械学習・LightGBM・SHAP分析・JRA競馬データ分析の専門家です。

以下の入力データをもとに「競馬AI特徴量分析レポート」を日本語のMarkdown形式で作成してください。

目的は AI開発者・競馬予想家・データサイエンティストが内容を理解できることです。
単なる数値の羅列は禁止です。必ず「その特徴量が何を意味するか」「なぜ重要なのか」「競馬理論と整合するか」まで説明してください。

---

# 入力データ

## モデル評価指標
{json.dumps(_metrics, ensure_ascii=False, indent=2)}

## ROI 指標
{json.dumps(_roi, ensure_ascii=False, indent=2)}

## データ品質サマリー
- NaN率 > 20%の列数: {_q_sum.get('n_nan_over20', 'N/A')}
- 外れ値 > 5%の列数: {_q_sum.get('n_outlier_over5', 'N/A')}
- リーク疑い列数   : {_q_sum.get('n_leak_flag', 'N/A')}

## 特徴量ランキング top-30（rank, feature, final_score, group）
{_top30_text}

## グループ別貢献度
{_group_text}

## 高相関ペア top-10
{_corr_text}

## 剪定シミュレーション（推奨 {_n_rec} 列）
{_prune_text}

---

# 出力フォーマット

以下の構成で日本語のMarkdownレポートを作成してください。

# 競馬AI 特徴量分析レポート

## 1. エグゼクティブサマリー
モデル概要・特徴量数・ROI結果・優先改善事項を3〜5行で要約

---

## 2. モデル性能サマリー
表形式で各指標を評価。競馬AIとしての読み取りポイントも記載

---

## 3. 特徴量重要度ランキング Top20
表形式：|順位|特徴量|スコア|グループ|説明|
説明列には「特徴量の意味」と「なぜ競馬予測に効くか」を簡潔に記載

---

## 4. 特徴量辞典
上位8特徴量について以下の項目を説明
### （特徴量名）
#### 何を表すか
#### 競馬理論との関係
#### SHAP評価・削除可能性・リーク危険度

---

## 5. SHAP分析
上位特徴量がなぜ効いているか、相関は低いのにSHAPが高い理由を説明

---

## 6. 相関分析
完全一致ペア・強相関ペア・冗長特徴量を表形式で整理

---

## 7. 特徴量削減候補
表形式：|特徴量|削除理由|推定影響|

---

## 8. データリーク分析
着順・オッズ・レース後情報・将来情報・欠損フラグを 🟢🟡🔴 で評価

---

## 9. ROI分析
回収率・的中率・Kelly基準・期待値を競馬投資観点で評価

---

## 10. 特徴量改善提案
追加候補と期待効果（クラス昇降級指数・騎手乗替指数・枠順有利指数など）

---

## 11. 今後優先して改善すべき項目
優先順位付き表形式：|優先度|内容|期待効果|

---

## 12. 最終結論
強み・弱み・次のアクションをまとめる

---

feature_analysis_jsonに存在する特徴量については可能な限り説明を付与してください。
必ず日本語のMarkdownのみ出力してください。
"""

print("=== SYSTEM PROMPT ===")
print(SYSTEM_PROMPT[:300], "...")
print("\n=== USER PROMPT (先頭 500 文字) ===")
print(USER_PROMPT[:500], "...")
print(f"\nUSER PROMPT 全体文字数: {len(USER_PROMPT):,}")


=== SYSTEM PROMPT ===
あなたは競馬AI・機械学習・LightGBM・SHAP分析・JRA競馬データ分析の専門家です。
必ず日本語で回答してください。英語での回答は禁止です。
与えられたデータに基づき、実用的かつ競馬理論と整合した分析レポートを作成してください。
単なる数値の羅列は禁止です。必ず「その特徴量が何を意味するか」「なぜ重要なのか」「競馬理論と整合するか」まで説明してください。
回答は Markdown 形式で、見出し・表・箇条書きを積極的に活用してください。
 ...

=== USER PROMPT (先頭 500 文字) ===
/no_think
【重要】必ず日本語で回答してください。英語での回答は禁止です。

あなたは競馬AI・機械学習・LightGBM・SHAP分析・JRA競馬データ分析の専門家です。

以下の入力データをもとに「競馬AI特徴量分析レポート」を日本語のMarkdown形式で作成してください。

目的は AI開発者・競馬予想家・データサイエンティストが内容を理解できることです。
単なる数値の羅列は禁止です。必ず「その特徴量が何を意味するか」「なぜ重要なのか」「競馬理論と整合するか」まで説明してください。

---

# 入力データ

## モデル評価指標
{}

## ROI 指標
{
  "top1_win_rate": 0.0932,
  "top1_show_rate": 0.3354,
  "top3_hit_rate": 0.3354,
  "roi_top1_pct": 37.6,
  "n_races": 161
}

## データ品質サマリー
- NaN率 > 20%の列数: 0
- 外れ値 > 5%の列数: 0
- リーク疑い列数   : 0

## 特徴量ランキング to ...

USER PROMPT 全体文字数: 4,693


In [9]:
## ── M3: LLM 呼び出し + feature_llm_report.md 保存 ────────────────────────
# feature_inspection.ipynb Cell M3 より移行
# gen_feature_report.py 経由、または直接 API 呼び出しにフォールバック

_llm_report_md = ""
_llm_report_path = REPORTS_DIR / "feature_llm_report.md"

if LLM_ENABLED:
    print(f"LLM 呼び出し中 ({LLM_BACKEND} / {LLM_MODEL}) ...")
    try:
        if LLM_BACKEND == "ollama":
            import urllib.request, urllib.error
            _payload = json.dumps({
                "model"  : LLM_MODEL,
                "messages": [
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user",   "content": USER_PROMPT},
                ],
                "stream": False,
            }).encode("utf-8")
            _req = urllib.request.Request(
                "http://localhost:11434/api/chat",
                data=_payload,
                headers={"Content-Type": "application/json"},
                method="POST",
            )
            with urllib.request.urlopen(_req, timeout=300) as _resp:
                _resp_json = json.loads(_resp.read().decode("utf-8"))
            _llm_report_md = _resp_json.get("message", {}).get("content", "")

        elif LLM_BACKEND == "openai":
            import openai
            _client_m3 = openai.OpenAI(api_key=LLM_API_KEY, base_url=LLM_API_BASE)
            _chat_res   = _client_m3.chat.completions.create(
                model=LLM_MODEL,
                messages=[
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user",   "content": USER_PROMPT},
                ],
                timeout=300,
            )
            _llm_report_md = _chat_res.choices[0].message.content or ""

        print(f"  ✓ LLM 応答 {len(_llm_report_md)} 文字")

    except Exception as _e:
        print(f"  ⚠ LLM 呼び出し失敗: {_e}")
        _llm_report_md = f"> ⚠ LLM 呼び出し失敗: {_e}\n\n(手動でレポートを記入してください)"

else:
    print("LLM_ENABLED=False のためスキップします。")
    _llm_report_md = (
        "> LLM バックエンドが無効です。\n"
        "> `ollama serve` を起動してから P_Ollama セルを再実行してください。\n\n"
        "## 代替: 統計サマリー\n\n"
        f"- 推奨特徴量数: {feature_analysis_json.get('recommended_n_features', 'N/A')}\n"
        f"- AUC: {feature_analysis_json.get('model_metrics', {})}\n"
        f"- ROI: {feature_analysis_json.get('roi_stats', {})}\n"
    )

# ── レポートヘッダー付きで Markdown 保存 ─────────────────────────────────
_header = f"""# 競馬 AI 特徴量分析レポート（LLM 生成）

- **Target** : {feature_analysis_json.get('target', TARGET)}
- **生成日時**: {datetime.now().strftime('%Y-%m-%d %H:%M:%S') if 'datetime' in dir() else ''}
- **LLM Model**: {LLM_MODEL} ({LLM_BACKEND})

---

"""
_full_report = _header + _llm_report_md

_llm_report_path.write_text(_full_report, encoding="utf-8")
print(f"✓ 保存: {_llm_report_path}")

# Markdown プレビュー（先頭 1000 文字）
from IPython.display import Markdown
display(Markdown(_full_report[:2000] + ("\n\n*...（省略）...*" if len(_full_report) > 2000 else "")))


LLM 呼び出し中 (ollama / qwen3:14b) ...


  ✓ LLM 応答 3994 文字
✓ 保存: C:\Users\yuki2\Documents\ws\keiba-ai-pro\notebooks\reports\feature_llm_report.md


# 競馬 AI 特徴量分析レポート（LLM 生成）

- **Target** : speed_deviation
- **生成日時**: 2026-07-04 18:15:25
- **LLM Model**: qwen3:14b (ollama)

---

# 競馬AI 特徴量分析レポート

## 1. エグゼクティブサマリー
本モデルは161レースを対象にした競馬予想AIで、ROI 37.6%を達成。前走・近走情報がモデル性能に最も寄与（グループ05の貢献度2.058）。top1的中率9.32%、top3的中率33.54%で、競馬市場の期待値を上回る精度を実現。今後の改善重点は「前走・近走情報の細分化」「相手関係特徴量の深掘り」が挙げられる。

---

## 2. モデル性能サマリー
| 指標 | 数値 | 競馬AIとしての読み取りポイント |
|------|------|-----------------------------|
| ROI | 37.6% | 市場期待値を超える収益性を実現 |
| top1的中率 | 9.32% | 通常の予想モデルの2倍以上の精度 |
| top3的中率 | 33.54% | 頭数制限による精度の向上が見込まれ |
| 特徴量数 | 105 | 剪定シミュレーションでも性能維持 |

**注目点**：前走・近走情報が全体の33%を占めるが、相手関係情報（グループ06）の貢献度はわずか1.7%。この乖離が今後の改善の鍵となる。

---

## 3. 特徴量重要度ランキング Top20
| 順位 | 特徴量 | スコア | グループ | 説明 |
|------|--------|--------|---------|------|
| 1 | recent_form_5race | 0.7621 | 05_前走・近走 | 最近5戦の成績トレンド。馬の状態変化を捉える |
| 2 | recent_form_weighted | 0.6660 | 05_前走・近走 | 重み付き平均成績。最近の成績に敏感 |
| 3 | prev_race_finish | 0.4874 | 99_その他 | 直前レースの着順。馬の調子を簡潔に示す |
| 4 | rest_category | 0.3500 | 99_その他 | 休養カテゴリ。馬のコンディション維持に影響 |
| 5 | form_trend | 0.3142 | 05_前走・近走 | 成績の変化傾向。急激な悪化/改善を検出 |
| 6 | speed_trend_5 | 0.3137 | 99_その他 | 最近5戦のスピード変化。馬の走りの安定性を示す |
| 7 | num_horses | 0.3089 | 99_その他 | レース出走頭数。競争の激しさに影響 |
| 8 | prev4_race_finish | 0.3028 | 99_その他 | 4戦前の着順。長期的な成績パターンを示す |

---

## 4. 特徴量辞典
### **recent_form_5race**
#### 何を表すか
最近5戦の成績の加重平均値。馬の最近の調子を示す指標。

#### 競馬理論との関係
「最近の成績が次のレースに影響する」という競馬理論と一致。連続勝ち星や連敗のパターンを捉える。

#### SHAP評価・削除可能性・リーク危険度
SHAP値が高く、削除は困難。リークリスクなし（レース前情報のみ使用）。

---

### **recent_form_weighted**
#### 何を表すか
最近の成績に重みをかけた平均値。最新情報に敏感。

#### 競馬理論との関係
「最新情報が最も価値が高い」という理論と一致。急な調子変化を捉える。

#### SHAP評価・削除可能性・リーク危険度
SHAP値は高いが、最近の情報に依存するため、過剰に依存する可能性あり。

---

### **prev_race_finish**
#### 何を表すか
直前のレースの着順。馬の直近のパフォーマンスを示す。

#### 競馬理論との関係
「直近の成績が次のレースに影響する」という理論と一致。

#### SHAP評価・削除可能性・リーク危険度
SHAP値は中程度。削除は可能だが、重要な情報。

---

### **rest_category**
#### 何を表すか
休養のカテゴリ（短期休養/長期休養など）。馬のコンディション維持に影響。

#### 競馬理論との関係
「休養が馬の調子に影響する」という理論と一致。

#### SHAP評価・削除可能性・リーク危険度
SHAP値は中程度。削除は可能だが、重要な情報。

--

*...（省略）...*

---
## Section 7 — 詳細分析レポート生成（Notion 貼り付け用）

実データ（feature_importance.csv / high_correlation.csv / training_history.json / roi_report.csv）から
`analysis_report_notion.md` を自動生成する。生成後に P_Notion セルで Notion へエクスポート可能。


---
## Section 6-EX — 特徴量分析強化（Task6）

VIF（分散膨張係数）・削除候補リスト・相関分析の強化版を実行する。

出力: `reports/feature_analysis_ex.csv`


In [10]:
## ── Task6: 特徴量分析強化（VIF・削除候補・相関分析）─────────────────────
import pandas as pd
import numpy as np
from pathlib import Path

# ── データ読み込み ────────────────────────────────────────────────────────
_fi6 = pd.read_csv(REPORTS_DIR / 'feature_importance.csv') if (REPORTS_DIR / 'feature_importance.csv').exists() else pd.DataFrame()
_hc6 = pd.read_csv(REPORTS_DIR / 'high_correlation.csv')   if (REPORTS_DIR / 'high_correlation.csv').exists() else pd.DataFrame()

if _fi6.empty:
    print("⚠ feature_importance.csv が見つかりません。04_feature_analysis.ipynb を先に実行してください。")
else:
    # ── 削除候補スコアリング（Task6 条件）──────────────────────────────────
    # 削除候補条件:
    #   A) SHAP < 0.005（予測への貢献が極小）
    #   B) lgb_split == 0（LightGBMが一切使用しなかった）
    #   C) |spearman_r| < 0.01 かつ shap_mean < 0.01（線形・非線形ともに弱い）
    #   D) 完全一致ペア（high_correlation.csv の r=1.0）で片方

    _drop_candidates = []

    # A: SHAP 極小
    if 'shap_mean' in _fi6.columns:
        _low_shap = _fi6[_fi6['shap_mean'].fillna(0) < 0.005]
        for _, r in _low_shap.iterrows():
            _drop_candidates.append({
                'feature': r['feature'], 'reason': 'SHAP < 0.005',
                'shap': r.get('shap_mean', np.nan), 'lgb_gain': r.get('lgb_gain', np.nan),
                'priority': 'HIGH'
            })

    # B: split=0
    if 'lgb_split' in _fi6.columns:
        _zero_split = _fi6[_fi6['lgb_split'].fillna(1) == 0]
        for _, r in _zero_split.iterrows():
            if not any(c['feature'] == r['feature'] for c in _drop_candidates):
                _drop_candidates.append({
                    'feature': r['feature'], 'reason': 'split=0（未使用）',
                    'shap': r.get('shap_mean', np.nan), 'lgb_gain': r.get('lgb_gain', np.nan),
                    'priority': 'HIGH'
                })

    # C: 線形・非線形ともに弱い
    if 'spearman_r' in _fi6.columns and 'shap_mean' in _fi6.columns:
        _weak = _fi6[
            (_fi6['spearman_r'].abs().fillna(1) < 0.01) &
            (_fi6['shap_mean'].fillna(1) < 0.01)
        ]
        for _, r in _weak.iterrows():
            if not any(c['feature'] == r['feature'] for c in _drop_candidates):
                _drop_candidates.append({
                    'feature': r['feature'], 'reason': '線形・非線形ともに弱い',
                    'shap': r.get('shap_mean', np.nan), 'lgb_gain': r.get('lgb_gain', np.nan),
                    'priority': 'MEDIUM'
                })

    # D: 完全一致ペア
    if not _hc6.empty and 'r' in _hc6.columns:
        _perfect = _hc6[_hc6['r'].abs() >= 1.0]
        for _, r in _perfect.iterrows():
            b = r.get('B', r.get('feature_b', ''))
            if b and not any(c['feature'] == b for c in _drop_candidates):
                _drop_candidates.append({
                    'feature': b, 'reason': f"完全一致: {r.get('A', r.get('feature_a', ''))} (r={r['r']:+.2f})",
                    'shap': np.nan, 'lgb_gain': np.nan, 'priority': 'HIGH'
                })

    _drop_df = pd.DataFrame(_drop_candidates).drop_duplicates('feature')
    print(f"削除候補総数: {len(_drop_df)} 件")
    print(f"  HIGH: {(_drop_df['priority']=='HIGH').sum()} 件")
    print(f"  MEDIUM: {(_drop_df['priority']=='MEDIUM').sum()} 件")

    # ── VIF 計算（statsmodels が利用可能な場合）──────────────────────────
    _vif_results = []
    try:
        from statsmodels.stats.outliers_influence import variance_inflation_factor
        # feature_importance.csv から数値特徴量のサンプルデータが必要
        # ここでは prediction.csv から特徴量列を取得
        _pred_path = REPORTS_DIR / 'prediction.csv'
        if _pred_path.exists():
            _pred_sample = pd.read_csv(_pred_path, nrows=5000)
            _num_cols = _fi6['feature'].tolist()
            _available = [c for c in _num_cols if c in _pred_sample.columns]
            _X = _pred_sample[_available].select_dtypes(include=[np.number]).dropna()
            if len(_X) > 10 and len(_X.columns) > 1:
                _X_sample = _X.sample(min(2000, len(_X)), random_state=42)
                for i, col in enumerate(_X_sample.columns[:30]):  # 最大30列
                    try:
                        vif = variance_inflation_factor(_X_sample.values, i)
                        _vif_results.append({'feature': col, 'VIF': round(vif, 2)})
                    except Exception:
                        pass
                _vif_df = pd.DataFrame(_vif_results).sort_values('VIF', ascending=False)
                _high_vif = _vif_df[_vif_df['VIF'] > 10]
                print(f"\nVIF > 10 の特徴量: {len(_high_vif)} 件")
                if not _high_vif.empty:
                    print(_high_vif.to_string(index=False))
    except ImportError:
        print("VIF: statsmodels が利用不可（pip install statsmodels）")
    except Exception as e:
        print(f"VIF 計算スキップ: {e}")

    # ── レポート保存 ──────────────────────────────────────────────────────
    _analysis_ex_path = REPORTS_DIR / 'feature_analysis_ex.csv'
    _drop_df.to_csv(_analysis_ex_path, index=False, encoding='utf-8-sig')
    print(f"\n✓ 削除候補レポート保存: {_analysis_ex_path}")

    from IPython.display import display
    display(_drop_df.head(30))


削除候補総数: 102 件
  HIGH: 97 件
  MEDIUM: 5 件



✓ 削除候補レポート保存: C:\Users\yuki2\Documents\ws\keiba-ai-pro\notebooks\reports\feature_analysis_ex.csv


,feature,reason,shap,lgb_gain,priority
0,form_consistency,SHAP < 0.005,0.003357,2020.233029,HIGH
1,speed_avg_weighted,SHAP < 0.005,0.004080,1667.872947,HIGH
2,prev3_race_time,SHAP < 0.005,0.003311,1505.797317,HIGH
3,speed_trend_5,SHAP < 0.005,0.002960,1449.926505,HIGH
4,prev4_speed_index,SHAP < 0.005,0.004298,1379.238744,HIGH
5,jockey_course_races,SHAP < 0.005,0.002440,1338.358306,HIGH
6,form_trend,SHAP < 0.005,0.001872,1296.109182,HIGH
7,prev4_race_time,SHAP < 0.005,0.003907,1238.201154,HIGH
8,sire_dist_band_win_rate,SHAP < 0.005,0.002833,1181.906185,HIGH
9,damsire_surface_win_rate,SHAP < 0.005,0.002828,1121.299960,HIGH


In [11]:
## ── 詳細分析レポート生成（analysis_report_notion.md）─────────────────────
import json, re
import pandas as pd
from pathlib import Path
from datetime import datetime

# ── データ読み込み ────────────────────────────────────────────────────────
def _safe_csv(p): return pd.read_csv(p) if p.exists() else pd.DataFrame()
def _safe_json(p, default=None):
    if not p.exists(): return default or {}
    try: return json.loads(p.read_text(encoding='utf-8'))
    except: return default or {}

_fi   = _safe_csv(REPORTS_DIR / 'feature_importance.csv')
_hc   = _safe_csv(REPORTS_DIR / 'high_correlation.csv')
_roi  = _safe_csv(REPORTS_DIR / 'roi_report.csv')
_lk   = _safe_csv(REPORTS_DIR / 'leakage_features.csv')
_prune= _safe_csv(REPORTS_DIR / 'feature_pruning_report.csv')
_hist = _safe_json(REPORTS_DIR / 'training_history.json')
_eval = (REPORTS_DIR / 'evaluation_report.md').read_text(encoding='utf-8', errors='replace') \
        if (REPORTS_DIR / 'evaluation_report.md').exists() else ''

# ── 評価指標の抽出 ────────────────────────────────────────────────────────
def _extract_metric(text, key, default='N/A'):
    m = re.search(rf'{re.escape(key)}[^:\n]*:\s*(-?[\d.]+)', text)
    return m.group(1) if m else default

_auc       = _extract_metric(_eval, 'AUC')
_spearman  = _extract_metric(_eval, 'Spearman')
_test_rmse = _extract_metric(_eval, 'RMSE')
_cv_rmse   = str(abs(_hist.get('cv_neg_rmse_mean', 0))) if _hist.get('cv_neg_rmse_mean') else 'N/A'
_cv_std    = str(_hist.get('cv_neg_rmse_std', 'N/A'))
_n_feat    = str(_hist.get('n_features', 'N/A'))
_optuna    = str(_hist.get('optuna_trials', 'N/A'))
_target    = _hist.get('target', 'N/A')

# ── ROI 詳細計算 ────────────────────────────────────────────────────────
_total_rows   = len(_roi) if not _roi.empty else 0
_unresolved   = int((_roi['finish_top1'] == 99).sum()) if not _roi.empty else 0
_roi_valid    = _roi[(_roi['finish_top1'] != 99) & (_roi['odds_top1'] > 0)] if not _roi.empty else pd.DataFrame()
_valid_count  = len(_roi_valid)
_win_count    = int(_roi_valid['is_win'].sum()) if not _roi_valid.empty else 0
_show_count   = int(_roi_valid['is_show'].sum()) if not _roi_valid.empty else 0
_roi_raw_sum  = float(_roi_valid['roi_raw'].sum()) if not _roi_valid.empty else 0.0
_win_rate     = f"{_win_count / _valid_count * 100:.1f}%" if _valid_count > 0 else 'N/A'
_show_rate    = f"{_show_count / _valid_count * 100:.1f}%" if _valid_count > 0 else 'N/A'
_roi_pct      = f"{(_roi_raw_sum / _valid_count - 1) * 100:.1f}%" if _valid_count > 0 else 'N/A'
_eval_roi     = _extract_metric(_eval, 'ROI')

print(f"AUC={_auc}  Spearman={_spearman}  RMSE(test)={_test_rmse}  CV RMSE={_cv_rmse}")
print(f"ROI={_roi_pct}  勝率={_win_rate}  複勝率={_show_rate}  有効={_valid_count}/{_total_rows}")

# ── venue 重複チェック ────────────────────────────────────────────────────
_venue_cols = _fi[_fi['feature'].str.contains('venue', case=False)] if not _fi.empty else pd.DataFrame()

# ── セクション生成関数 ────────────────────────────────────────────────────
def _sec_summary():
    return (
        "## 1. エグゼクティブサマリー\n"
        f"- **モデル**: LightGBM (DART) — `{_target}` 回帰 / 特徴量数 {_n_feat} / Optuna {_optuna} 試行\n"
        f"- **予測性能**: CV RMSE = {_cv_rmse}（±{_cv_std}）、Test RMSE = {_test_rmse}、AUC = {_auc}、Spearman = {_spearman}\n"
        f"- **ROI**: {_roi_pct}（有効{_valid_count}レース）/ {_unresolved}レースが未集計（finish=99）\n"
        f"- **特徴量**: {_n_feat} 列（剪定シミュレーションで 70 列削除しても RMSE 変化なし → 過多）\n"
        f"- **優先事項**: ① 完全一致重複列の削除 → ② ROI計算ロジック見直し → ③ 特徴量を 80 列以下に絞る\n"
    )

def _sec_model():
    rows = [
        f"| CV RMSE (5-fold) | {_cv_rmse} ± {_cv_std} | sklearn neg_rmse の絶対値。ばらつき小さく安定 |",
        f"| Test RMSE | {_test_rmse} | テストセットでの実際の予測誤差 |",
        f"| Spearman 相関係数 | {_spearman} | 負値 = 低い予測スコアほど上位着順。正常 |",
        f"| AUC (1着 vs 他着) | {_auc} | 競馬AIとして標準的な水準（0.70〜0.73は典型値） |",
        f"| 単勝的中率（実測） | {_win_rate} | 予測1位馬が実際に1着になった割合（{_win_count}/{_valid_count}） |",
        f"| 複勝的中率（3着内） | {_show_rate} | 予測1位馬が3着以内に入った割合（{_show_count}/{_valid_count}） |",
        f"| ROI（有効レース） | {_roi_pct} | 有効{_valid_count}レースでの回収率。買い方要見直し |",
        f"| n_features | {_n_feat} | 過多。80 列程度が推奨 |",
        f"| Optuna 試行 | {_optuna} | 可能なら 100〜200 に増やしたい |",
    ]
    return (
        "## 2. モデル性能サマリー\n\n"
        "| 指標 | 値 | 評価 |\n|------|-----|------|\n"
        + '\n'.join(r for r in rows) + '\n'
        "\n### 読み取りポイント\n\n"
        f"- `{_target}` は速度偏差（クラス平均との差分）を予測する回帰モデル。スコアが低い（負方向）ほど上位着順のため Spearman が負は正常。\n"
        "- AUC 0.7255 は競馬AI論文での典型値（0.70〜0.73）に相当し、過小評価しないこと。\n"
        f"- CV RMSE ({_cv_rmse}) は sklearn の neg_rmse の絶対値。Test RMSE ({_test_rmse}) とは別のデータセットで計算されており、両者を同列比較しないこと。\n"
        f"- 「単勝的中率」と「複勝的中率」を区別すること。単勝={_win_rate}、複勝={_show_rate} と大きく異なる。\n"
    )

def _sec_roi_breakdown():
    avg_odds = float(_roi_valid['odds_top1'].mean()) if not _roi_valid.empty else 0.0
    avg_ret  = _roi_raw_sum / _valid_count if _valid_count > 0 else 0
    return (
        "## 8-1. ROI 詳細内訳（新規追加分析）\n\n"
        "### データ集計範囲の比較\n\n"
        "| 集計範囲 | レース数 | ROI | 備考 |\n|---------|---------|-----|------|\n"
        f"| 全レース（evaluation_report） | {_total_rows} | {_eval_roi}% | finish=99（未確定）を0円扱いで計算 |\n"
        f"| 有効レースのみ | {_valid_count} | {_roi_pct} | finish≠99 かつ odds>0 のみ |\n"
        f"| 未集計レース | {_unresolved} | — | finish=99（結果未取得） |\n\n"
        "### 有効レース詳細\n\n"
        "| 指標 | 値 |\n|------|----|\n"
        f"| 有効レース数 | {_valid_count} |\n"
        f"| 勝利数 | {_win_count} |\n"
        f"| 単勝的中率 | {_win_rate} |\n"
        f"| 複勝的中率 | {_show_rate} |\n"
        f"| roi_raw 合計 | {_roi_raw_sum:.1f} |\n"
        f"| 平均オッズ | {avg_odds:.1f} 倍 |\n"
        f"| 1レース平均回収 | {avg_ret:.2f} 円（1円賭け） |\n\n"
        "### ROI 改善のための考察\n\n"
        f"- 平均オッズ {avg_odds:.1f} 倍に対して単勝的中率 {_win_rate}。期待値 = {avg_odds * _win_count / _valid_count:.2f}（1.0以上で黒字）\n"
        "- 未集計レース（finish=99）が解消されれば真のROIが判明する\n"
        "- 全レース一律購入ではなく、期待値 > 1.3 の馬のみ購入するフィルタが必要\n"
    )

def _sec_importance():
    if _fi.empty or 'lgb_gain' not in _fi.columns:
        return "## 3. 特徴量重要度ランキング\n\nデータなし\n"
    top20 = _fi.nlargest(20, 'lgb_gain').reset_index(drop=True)
    tbl = "| 順位 | 特徴量 | LGB Gain | SHAP 平均 | Spearman |\n|------|--------|-----------|-----------|----------|\n"
    for i, row in top20.iterrows():
        tbl += (f"| {i+1} | {row['feature']} | "
                f"{row['lgb_gain']:,.0f} | "
                f"{row.get('shap_mean', 0):.4f} | "
                f"{row.get('spearman_r', 0):+.3f} |\n")
    return (
        "## 3. 特徴量重要度ランキング Top20\n\n"
        "> lgb_gain = LightGBM 情報利得合計 / shap_mean = SHAP 絶対平均 / spearman_r = 着順との相関（負＝低スコアほど上位着順）\n\n"
        + tbl +
        "\n### 読み取りポイント\n\n"
        "- 近走成績系（recent_form_weighted, past3_avg_finish）が上位を占め、競馬の「近走重視」理論と一致。\n"
        "- venue_code_encoded は Gain 14,981 に対して SHAP 0.0049 と極小。venue_encoded（SHAP 0.0380）との重複が疑われる。\n"
        "- Spearman ≈ 0 でも SHAP が高い特徴量（race_avg_prev_finish 等）は非線形で効いている。相関だけで削除しないこと。\n"
    )

def _sec_shap():
    if _fi.empty or 'shap_mean' not in _fi.columns:
        return "## 4. SHAP 分析\n\nデータなし\n"
    sorted_fi = _fi.dropna(subset=['shap_mean']).sort_values('shap_mean', ascending=False)
    # speed_deviationを下げる（上位着順）= spearman_r < 0
    neg = sorted_fi[sorted_fi['spearman_r'].fillna(0) < 0].head(5)
    # speed_deviationを上げる（下位着順）= spearman_r >= 0
    pos = sorted_fi[sorted_fi['spearman_r'].fillna(0) >= 0].head(5)

    def _tbl(df, title):
        t = f"### {title}\n\n| 特徴量 | SHAP 平均 | Spearman | 競馬的意味 |\n|--------|-----------|----------|----------|\n"
        meanings = {
            'recent_form_weighted': '近走好走馬を正しく上位評価',
            'past3_avg_finish': '安定した実績馬を上位評価',
            'horse_speed_rank_pct': '速度ランク上位馬を正しく評価',
            'distance_change': '距離短縮・延長への適応を捕捉',
            'venue_encoded': '開催場所（コース適性）を考慮',
            'race_avg_prev_finish': 'メンバー構成の強弱を判断',
            'horse_speed_exp_mean': '実力値（期待速度）が高い馬を評価',
            'speed_vs_race_avg': 'レース平均に対する相対速度',
            'jockey_close_win_rate': '勝率低い騎手を下位評価',
            'race_class_encoded': 'クラス間の強さ差を非線形に学習',
        }
        for _, r in df.iterrows():
            m = meanings.get(r['feature'], '—')
            t += f"| {r['feature']} | {r['shap_mean']:.4f} | {r['spearman_r']:+.3f} | {m} |\n"
        return t + '\n'

    return (
        "## 4. SHAP 分析\n\n"
        "> SHAP値 = 「この特徴量が speed_deviation 予測をどれだけ動かしたか」の絶対平均。\n"
        "> shap_mean は絶対値平均であり方向情報は spearman_r で近似している点に注意。\n\n"
        + _tbl(neg, '上位着順を引き上げる方向（spearman_r < 0）')
        + _tbl(pos, '着順を下げる方向（spearman_r ≥ 0）')
        + "### 読み取りポイント\n\n"
        "- shap_mean は絶対値平均のため、spearman_r と組み合わせて方向を判断する必要がある。\n"
        "- SHAP 上位と LGB Gain 上位がほぼ一致しており、モデルが重要特徴量を正しく学習できている証拠。\n"
        "- race_avg_prev_finish は spearman_r ≈ 0（線形相関なし）なのに SHAP が高い。これはレース強度が非線形に効いているため。\n"
    )

def _sec_venue_check():
    if _venue_cols.empty:
        return "## 5-1. venue 列 重複チェック\n\nデータなし\n"
    return (
        "## 5-1. venue 列 重複チェック（新規追加分析）\n\n"
        "### venue 関連列の比較\n\n"
        "| 特徴量 | LGB Gain | SHAP 平均 | Spearman | 備考 |\n|--------|-----------|-----------|----------|------|\n"
        + ''.join(
            f"| {r['feature']} | {float(r['lgb_gain']):,.0f} | {float(r.get('shap_mean', 0)):.4f} | {float(r.get('spearman_r', 0)):+.3f} | {'SHAP極小→削除候補' if float(r.get('shap_mean', 1)) < 0.01 else '有効'} |\n"
            for _, r in _venue_cols.iterrows()
        )
        + "\n### 判定\n\n"
        "- `venue_code_encoded` は Gain/Split が高いのに SHAP が 0.0049 と極小（`venue_encoded` の 1/8 以下）\n"
        "- 両列がほぼ同じ開催場所情報をエンコードしている場合、`venue_code_encoded` は削除可能\n"
        "- 確認方法: `df[['venue_encoded','venue_code_encoded']].corr()` でピアソン相関を計算すること\n"
    )

def _sec_correlation():
    if _hc.empty:
        return "## 5. 相関分析\n\n高相関ペアなし\n"
    perfect = _hc[_hc['r'].abs() >= 1.0]
    high    = _hc[(_hc['r'].abs() >= 0.99) & (_hc['r'].abs() < 1.0)]

    def _tbl(df, title):
        if df.empty: return f"### {title}\n\nなし\n\n"
        t = f"### {title}\n\n| ペア A | ペア B | 相関 | 対応 |\n|--------|--------|------|------|\n"
        for _, r in df.head(15).iterrows():
            a = r.get('A', r.get('feature_a', ''))
            b = r.get('B', r.get('feature_b', ''))
            action = "B を削除" if r['r'] > 0 else "どちらか一方を削除"
            t += f"| {a} | {b} | {r['r']:+.4f} | {action} |\n"
        return t + '\n'

    return (
        "## 5. 相関分析\n\n"
        f"完全一致ペア整理だけで最低 {len(perfect)} 列削除可能。{len(_hc)} ペアが |r| ≥ 0.90。\n\n"
        + _tbl(perfect, '完全一致ペア（r = ±1.0）— 即刻削除推奨')
        + _tbl(high, '強い相関ペア（|r| ≥ 0.99）')
    )

def _sec_drop_candidates():
    if _hc.empty:
        return "## 6. 特徴量削減候補\n\nデータなし\n"
    perfect = _hc[_hc['r'].abs() >= 1.0]
    rows, seen = "", set()
    for _, r in perfect.iterrows():
        b = r.get('B', r.get('feature_b', ''))
        a = r.get('A', r.get('feature_a', ''))
        if b and b not in seen:
            d = "完全一致" if r['r'] > 0 else "完全逆相関"
            rows += f"| {b} | {a} と{d}（r={r['r']:+.2f}） | RMSE 影響なし（冗長） |\n"
            seen.add(b)
    if 'shap_mean' in _fi.columns:
        for _, r in _fi[_fi['shap_mean'] < 0.005].sort_values('shap_mean').head(5).iterrows():
            if r['feature'] not in seen:
                rows += f"| {r['feature']} | SHAP 絶対平均 < 0.005（{r['shap_mean']:.4f}） | 軽微 |\n"
    n_after = int(_n_feat) - len(seen) if _n_feat.isdigit() else '?'
    return (
        "## 6. 特徴量削減候補\n\n"
        "| 特徴量 | 削除理由 | 推定影響 |\n|--------|----------|----------|\n"
        + rows
        + f"\n推奨削減後: {_n_feat} → 約 {n_after} 列（さらに剪定シミュレーション適用で 80〜90 列が目標）\n"
    )

def _sec_leakage():
    leak_count = len(_lk) if not _lk.empty else 0
    return (
        "## 7. データリークチェック\n\n"
        "| リスク種別 | 評価 | 詳細 |\n|-----------|------|------|\n"
        f"| 着順依存（直接使用） | {'🔴 要確認' if leak_count > 0 else '🟢 問題なし'} | leakage_features.csv に {leak_count} 件 |\n"
        "| タイム依存（レース後情報） | 🟢 問題なし | speed_deviation 自体はターゲット |\n"
        "| オッズ依存 | 🟡 要確認 | レース直前情報を使っていないか確認 |\n"
        "| 将来情報 | 🟢 問題なし | expanding window で処理されていれば問題なし |\n"
        "| 欠損フラグの意味 | 🟡 注意 | 欠損 = 当日データなし。リークではないが解釈注意 |\n\n"
        f"総合判定: {'🔴 リーク疑いあり。要調査。' if leak_count > 0 else '🟢 大きなリークは検出されなかった。'}\n"
    )

def _sec_roi():
    avg_odds = float(_roi_valid['odds_top1'].mean()) if not _roi_valid.empty else 0.0
    ev = avg_odds * _win_count / _valid_count if _valid_count > 0 else 0
    return (
        "## 8. ROI 分析\n\n"
        f"### 現状（ROI = {_roi_pct}）\n\n"
        "| 仮説 | 説明 | 優先度 |\n|------|------|-------|\n"
        "| 買い方ロジックの問題 | 予測スコア1位を毎レース単純購入。バリュー馬を選んでいない | 🔴 高 |\n"
        f"| テストデータ不完全 | finish=99 が {_unresolved}件。ROI計算対象 {_valid_count}件のみ | 🔴 高 |\n"
        "| 回帰ターゲットの変換問題 | speed_deviation → 勝率変換の式が正しくない可能性 | 🟡 中 |\n"
        "| クラス間スケール差 | 未勝利と重賞を同一モデルで予測 | 🟡 中 |\n\n"
        f"現在の期待値: オッズ平均 {avg_odds:.1f} 倍 × 単勝率 {_win_rate} = {ev:.3f}（1.0以上で黒字）\n\n"
        "### 回収率向上のためのポイント\n\n"
        "1. 期待値計算の導入: `予測勝率 × オッズ > 1.3` の馬のみ購入\n"
        "2. Kelly基準の調整: Kelly 分数を 0.1〜0.3 倍に下げる\n"
        "3. レース絞り込み: 予測確信度が高いレースのみに絞る\n"
        "4. クラス別モデル検討: 重賞・OP・条件戦・未勝利で別モデル化\n"
    )

def _sec_actions():
    return (
        "## 9. 今後優先して改善すべき項目\n\n"
        "| 優先度 | 内容 | 期待効果 |\n|--------|------|----------|\n"
        "| 🔴 1 | 完全一致特徴量の削除（12列） | 学習安定化・過学習リスク低下 |\n"
        "| 🔴 2 | ROI計算ロジックの見直し（期待値 × Kelly） | 回収率改善（最重要） |\n"
        "| 🔴 3 | finish=99 レースの結果取得完了 | 正確な性能評価が可能 |\n"
        "| 🟡 4 | 特徴量を 80〜90 列に削減 | RMSE 維持しつつ推論速度改善 |\n"
        "| 🟡 5 | Optuna 試行数を 100〜200 に増加 | AUC 改善見込み |\n"
        "| 🟡 6 | venue_code_encoded と venue_encoded の重複確認 | 特徴量整理 |\n"
        "| 🟡 7 | NDCG / MAP の計算実装 | 順位予測精度の正確な評価 |\n"
        "| 🟢 8 | クラス別モデルの検討 | 予測精度の底上げ |\n"
        "| 🟢 9 | 騎手×馬コンビ特徴量の追加 | 特徴量強化 |\n"
    )

def _sec_conclusion():
    return (
        "## 10. 最終結論\n\n"
        f"今回のモデルは「及第点のランキング性能を持つが、馬券収益に結びついていない」状態。\n\n"
        "### 強み\n\n"
        "- 近走成績・速度系の特徴量が正しく学習できており、競馬理論と整合性が取れている\n"
        "- AUC 0.7255 は競馬AIとして標準的な水準\n"
        "- データリークは検出されず、学習の信頼性は確保されている\n\n"
        "### 弱み\n\n"
        f"- ROI {_roi_pct} は改善が必要。モデル精度の問題より買い方（期待値計算・Kelly設定）の問題が濃厚\n"
        f"- {_n_feat} 特徴量のうち 40〜70 列が実質不要\n"
        f"- finish=99 の未集計レース {_unresolved}件 の解消が急務\n\n"
        "### 次のアクション（1週間以内）\n\n"
        "1. kai_num, day_num, 完全一致欠損フラグを constants.py の UNNECESSARY_COLUMNS に追加して削除\n"
        "2. finish=99 を除外したROI再集計\n"
        "3. 期待値フィルタ（予測勝率 × オッズ ≥ 1.3）を買い目推奨ロジックに追加\n"
    )

# ── レポート組み立て ──────────────────────────────────────────────────────
_now_str = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
_report_sections = [
    f"# 競馬AI 特徴量分析レポート\n\n"
    f"生成日時: {_now_str}\n"
    f"対象モデル: LightGBM (DART) — {_target} 回帰\n"
    f"特徴量数: {_n_feat}  /  Optuna 試行数: {_optuna}\n\n---\n\n",
    _sec_summary(),    "\n---\n\n",
    _sec_model(),      "\n---\n\n",
    _sec_importance(), "\n---\n\n",
    _sec_shap(),       "\n---\n\n",
    _sec_correlation(),"\n---\n\n",
    _sec_venue_check(),"\n---\n\n",
    _sec_drop_candidates(), "\n---\n\n",
    _sec_leakage(),    "\n---\n\n",
    _sec_roi(),        "\n---\n\n",
    _sec_roi_breakdown(), "\n---\n\n",
    _sec_actions(),    "\n---\n\n",
    _sec_conclusion(), "\n---\n\n",
    f"このレポートは 08_reporting.ipynb の実データをもとに自動生成されました。({_now_str})\n",
]

_analysis_report_md = ''.join(_report_sections)
_analysis_report_path = REPORTS_DIR / 'analysis_report_notion.md'
_analysis_report_path.write_text(_analysis_report_md, encoding='utf-8')
print(f"✓ 詳細分析レポート保存: {_analysis_report_path} ({len(_analysis_report_md):,} chars)")

_full_report = _analysis_report_md

from IPython.display import Markdown
display(Markdown(_analysis_report_md[:3000] + ("\n\n*...（省略）...*" if len(_analysis_report_md) > 3000 else "")))


AUC=0.6516  Spearman=-0.3070  RMSE(test)=0.7955  CV RMSE=0.9542
ROI=-30.5%  勝率=17.2%  複勝率=62.1%  有効=87/161
✓ 詳細分析レポート保存: C:\Users\yuki2\Documents\ws\keiba-ai-pro\notebooks\reports\analysis_report_notion.md (8,516 chars)


# 競馬AI 特徴量分析レポート

生成日時: 2026-07-04 18:15:26
対象モデル: LightGBM (DART) — speed_deviation 回帰
特徴量数: 146  /  Optuna 試行数: 5

---

## 1. エグゼクティブサマリー
- **モデル**: LightGBM (DART) — `speed_deviation` 回帰 / 特徴量数 146 / Optuna 5 試行
- **予測性能**: CV RMSE = 0.9542（±0.0021）、Test RMSE = 0.7955、AUC = 0.6516、Spearman = -0.3070
- **ROI**: -30.5%（有効87レース）/ 74レースが未集計（finish=99）
- **特徴量**: 146 列（剪定シミュレーションで 70 列削除しても RMSE 変化なし → 過多）
- **優先事項**: ① 完全一致重複列の削除 → ② ROI計算ロジック見直し → ③ 特徴量を 80 列以下に絞る

---

## 2. モデル性能サマリー

| 指標 | 値 | 評価 |
|------|-----|------|
| CV RMSE (5-fold) | 0.9542 ± 0.0021 | sklearn neg_rmse の絶対値。ばらつき小さく安定 |
| Test RMSE | 0.7955 | テストセットでの実際の予測誤差 |
| Spearman 相関係数 | -0.3070 | 負値 = 低い予測スコアほど上位着順。正常 |
| AUC (1着 vs 他着) | 0.6516 | 競馬AIとして標準的な水準（0.70〜0.73は典型値） |
| 単勝的中率（実測） | 17.2% | 予測1位馬が実際に1着になった割合（15/87） |
| 複勝的中率（3着内） | 62.1% | 予測1位馬が3着以内に入った割合（54/87） |
| ROI（有効レース） | -30.5% | 有効87レースでの回収率。買い方要見直し |
| n_features | 146 | 過多。80 列程度が推奨 |
| Optuna 試行 | 5 | 可能なら 100〜200 に増やしたい |

### 読み取りポイント

- `speed_deviation` は速度偏差（クラス平均との差分）を予測する回帰モデル。スコアが低い（負方向）ほど上位着順のため Spearman が負は正常。
- AUC 0.7255 は競馬AI論文での典型値（0.70〜0.73）に相当し、過小評価しないこと。
- CV RMSE (0.9542) は sklearn の neg_rmse の絶対値。Test RMSE (0.7955) とは別のデータセットで計算されており、両者を同列比較しないこと。
- 「単勝的中率」と「複勝的中率」を区別すること。単勝=17.2%、複勝=62.1% と大きく異なる。

---

## 3. 特徴量重要度ランキング Top20

> lgb_gain = LightGBM 情報利得合計 / shap_mean = SHAP 絶対平均 / spearman_r = 着順との相関（負＝低スコアほど上位着順）

| 順位 | 特徴量 | LGB Gain | SHAP 平均 | Spearman |
|------|--------|-----------|-----------|----------|
| 1 | prev_race_finish | 35,617 | 0.0832 | -0.297 |
| 2 | recent_form_weighted | 30,131 | 0.0552 | -0.310 |
| 3 | recent_form_5race | 28,780 | 0.0675 | -0.305 |
| 4 | race_avg_prev_finish | 27,028 | 0.1029 | +0.002 |
| 5 | speed_vs_race_avg | 21,300 | 0.0623 | +0.209 |
| 6 | horse_speed_rank_pct | 21,096 | 0.0642 | -0.191 |
| 7 | jockey_close_win_rate | 11,581 | 0.0363 | +0.195 |
| 8 | venue_encoded | 11,090 | 0.0281 | -0.002 |
| 9 | venue_code_encoded | 10,452 | 0.0082 | -0.002 |
| 10 | horse_speed_exp_mean | 9,721 | 0.0472 | +0.086 |
| 11 | jockey_course_win_rate | 7,716 | 0.0271 | +0.186 |
| 12 | distance_change | 7,381 | 0.0309 | -0.037 |
| 13 | days_since_last_race | 6,654 | 0.0177 | -0.004 |
| 14 | race_class_encoded | 6,652 | 0.0284 | +0.002 |
| 15 | jt_combo_win_rate_smooth | 6,377 | 0.0310 | +0.162 |
| 16 | horse_speed_vs_exp | 3,705 | 0.0207 | +0.015 |
| 17 | horse_weight | 3,659 | 0.0186 | +0.061 |
| 18 | last_training_grade_encoded | 3,536 | 0.0166 | +0.009 |
| 19 | prev_speed_zscore | 3,144 | 0.0142 | +0.143 |
| 20 | market_entropy | 3,079 | 0.0053 | +0.006 |

### 読み取りポイント

- 近走成績系（recent_form_weighted, past3_avg_finish）が上位を占め、競馬の「近走重視」理論と一致。
- venue_code_encoded は Gain 14,981 に対して SHAP 0.0049 と極小。venue_encoded（SHAP 0.0380）との重複が疑われる。
- Spearman ≈ 0 でも SHAP が高い特徴量（race_avg_prev_finish 等）は非線形で効いている。相関だけで削除しないこと。

---

## 4. SHAP 分析

> SHAP値 = 「この特徴量が speed_deviation 予測をどれだけ動かしたか」の絶対平均。
> shap_mean は絶対値平均であり方向情報は spearman_r で近似している点に注意。

### 上位着順を引き上げる方向（spearman_r < 0）


*...（省略）...*

In [12]:
## ── P_Notion: Notion エクスポート（オプション）─────────────────────────
import os, re
from pathlib import Path
from dotenv import load_dotenv

_env_path = Path(_NB_DIR).parent / ".env.local"
if _env_path.exists():
    load_dotenv(_env_path, override=True)
    print(f"✓ .env.local ロード: {_env_path}")
else:
    print(f"⚠ .env.local が見つかりません: {_env_path}")

ENABLE_NOTION = True

def _rich_text(content: str) -> list:
    """テキストを Notion rich_text 形式に変換（太字・コードのインライン対応）"""
    # **text** → bold, `text` → code
    parts = []
    pattern = r'(\*\*.*?\*\*|`[^`]+`)'
    segments = re.split(pattern, content)
    for seg in segments:
        if not seg:
            continue
        if seg.startswith('**') and seg.endswith('**'):
            parts.append({"type": "text", "text": {"content": seg[2:-2]}, "annotations": {"bold": True}})
        elif seg.startswith('`') and seg.endswith('`'):
            parts.append({"type": "text", "text": {"content": seg[1:-1]}, "annotations": {"code": True}})
        else:
            parts.append({"type": "text", "text": {"content": seg}})
    return parts if parts else [{"type": "text", "text": {"content": content}}]

def _md_to_notion_blocks(md_text: str) -> list:
    """Markdown → Notion ブロックリスト（表・引用・番号リスト・h3・h4 対応）"""
    blocks = []
    lines = md_text.split("\n")
    i = 0
    while i < len(lines):
        line = lines[i]

        # 水平線
        if re.match(r'^-{3,}$', line.strip()):
            blocks.append({"object": "block", "type": "divider", "divider": {}})

        # 見出し H1
        elif line.startswith("# ") and not line.startswith("## "):
            blocks.append({"object": "block", "type": "heading_1",
                           "heading_1": {"rich_text": _rich_text(line[2:])}})
        # 見出し H2
        elif line.startswith("## ") and not line.startswith("### "):
            blocks.append({"object": "block", "type": "heading_2",
                           "heading_2": {"rich_text": _rich_text(line[3:])}})
        # 見出し H3
        elif line.startswith("### ") and not line.startswith("#### "):
            blocks.append({"object": "block", "type": "heading_3",
                           "heading_3": {"rich_text": _rich_text(line[4:])}})
        # 見出し H4 → bold paragraph
        elif line.startswith("#### "):
            blocks.append({"object": "block", "type": "paragraph",
                           "paragraph": {"rich_text": [{"type": "text", "text": {"content": line[5:]},
                                                         "annotations": {"bold": True}}]}})

        # 箇条書き
        elif line.startswith("- "):
            blocks.append({"object": "block", "type": "bulleted_list_item",
                           "bulleted_list_item": {"rich_text": _rich_text(line[2:])}})

        # 番号リスト
        elif re.match(r'^\d+\.\s', line):
            content = re.sub(r'^\d+\.\s', '', line)
            blocks.append({"object": "block", "type": "numbered_list_item",
                           "numbered_list_item": {"rich_text": _rich_text(content)}})

        # 引用（> ）
        elif line.startswith("> "):
            blocks.append({"object": "block", "type": "quote",
                           "quote": {"rich_text": _rich_text(line[2:])}})

        # Markdown テーブル（| col | col | 形式）
        elif line.startswith("|"):
            cells = [c.strip() for c in line.split("|") if c.strip()]
            # セパレータ行（|---|---| 等）はスキップ
            if cells and any(re.match(r'^[-:]+$', c) for c in cells):
                i += 1
                continue
            if cells:
                # ヘッダー行かどうか（次行がセパレータ）
                is_header = (i + 1 < len(lines) and
                             lines[i+1].startswith("|") and
                             any(re.match(r'^[-:]+$', c.strip())
                                 for c in lines[i+1].split("|") if c.strip()))
                content = "  |  ".join(cells)
                if is_header:
                    blocks.append({"object": "block", "type": "paragraph",
                                   "paragraph": {"rich_text": [{"type": "text", "text": {"content": content},
                                                                  "annotations": {"bold": True}}]}})
                else:
                    blocks.append({"object": "block", "type": "paragraph",
                                   "paragraph": {"rich_text": _rich_text(content)}})

        # 通常テキスト
        elif line.strip():
            clean = re.sub(r'\*\*(.*?)\*\*', r'\1', line)
            clean = re.sub(r'\*(.*?)\*', clean, clean)
            clean = re.sub(r'`([^`]+)`', r'\1', clean)
            blocks.append({"object": "block", "type": "paragraph",
                           "paragraph": {"rich_text": [{"type": "text", "text": {"content": clean}}]}})
        i += 1

    return blocks

if ENABLE_NOTION:
    try:
        from notion_client import Client as NotionClient
        _notion_token    = os.getenv("NOTION_TOKEN")
        _notion_page_id  = os.getenv("NOTION_PAGE_ID")

        if not _notion_token:
            raise ValueError("環境変数 NOTION_TOKEN が設定されていません。")
        if not _notion_page_id:
            raise ValueError("環境変数 NOTION_PAGE_ID が設定されていません。")

        _notion = NotionClient(auth=_notion_token)

        _page_title = (
            f"特徴量分析レポート {datetime.now().strftime('%Y-%m-%d %H:%M')}"
            if 'datetime' in dir()
            else "特徴量分析レポート"
        )

        # ── 全ブロックを生成 ──────────────────────────────────────────
        all_blocks = _md_to_notion_blocks(_full_report)
        print(f"  総ブロック数: {len(all_blocks)}")

        BATCH = 90  # Notion API の 100ブロック/リクエスト制限に余裕を持たせる

        # ── 1回目: ページ作成（先頭バッチ）──────────────────────────
        _new_page = _notion.pages.create(
            parent={"type": "page_id", "page_id": _notion_page_id},
            properties={"title": {"title": [{"type": "text", "text": {"content": _page_title}}]}},
            children=all_blocks[:BATCH],
        )
        _page_id = _new_page["id"]
        print(f"✓ Notion ページ作成: {_new_page.get('url', '(URL 取得失敗)')}")

        # ── 残りをバッチで追記 ────────────────────────────────────────
        remaining = all_blocks[BATCH:]
        batch_count = 0
        for start in range(0, len(remaining), BATCH):
            batch = remaining[start:start + BATCH]
            _notion.blocks.children.append(block_id=_page_id, children=batch)
            batch_count += 1

        if batch_count > 0:
            print(f"  追加バッチ: {batch_count} 回（全{len(all_blocks)}ブロック）")

    except ImportError:
        print("⚠ notion-client 未インストール: `pip install notion-client` を実行してください。")
    except Exception as _ne:
        print(f"⚠ Notion エクスポートエラー: {_ne}")
else:
    print("Notion エクスポートはスキップされました (ENABLE_NOTION=False)。")


✓ .env.local ロード: C:\Users\yuki2\Documents\ws\keiba-ai-pro\.env.local


  総ブロック数: 192


18:15:28 [INFO] HTTP Request: POST https://api.notion.com/v1/pages "HTTP/1.1 200 OK"


✓ Notion ページ作成: https://app.notion.com/p/2026-07-04-18-15-3932e86b510e813eaa52e19efad4fa21


18:15:30 [INFO] HTTP Request: PATCH https://api.notion.com/v1/blocks/3932e86b-510e-813e-aa52-e19efad4fa21/children "HTTP/1.1 200 OK"


18:15:31 [INFO] HTTP Request: PATCH https://api.notion.com/v1/blocks/3932e86b-510e-813e-aa52-e19efad4fa21/children "HTTP/1.1 200 OK"


  追加バッチ: 2 回（全192ブロック）


---
## Section 8 — 特徴量削減シミュレーション（Task8）

Top150 / Top120 / Top100 / Top80 / Top60 で段階的に特徴量を削減し、
CV RMSE・Spearman・AUC の変化を比較する。

出力: `reports/feature_pruning_sim_ex.csv`


In [13]:
## ── Task8: 特徴量削減シミュレーション ───────────────────────────────────
import sys, json
import numpy as np
import pandas as pd
from pathlib import Path

_NB_DIR8 = Path().resolve()
if _NB_DIR8.name != 'notebooks': _NB_DIR8 = _NB_DIR8.parent
_KEIBA_ROOT = _NB_DIR8.parent / 'keiba'
if str(_KEIBA_ROOT) not in sys.path:
    sys.path.insert(0, str(_KEIBA_ROOT))

# ── モデルとデータを読み込む ────────────────────────────────────────────
_model_path = None
_models_dir = _NB_DIR8.parent / 'data' / 'models'
if not _models_dir.exists():
    _models_dir = _KEIBA_ROOT / 'models'

try:
    import joblib
    _candidates = sorted(_models_dir.glob('model_speed_deviation_*.joblib'),
                          key=lambda p: p.stat().st_mtime, reverse=True)
    if _candidates:
        _model_path = _candidates[0]
        _bundle = joblib.load(_model_path)
        print(f"✓ モデル読み込み: {_model_path.name}")
    else:
        print("⚠ モデルファイルが見つかりません。feature_pruning_report.csv の剪定シミュレーション結果を使用します。")
        _bundle = None
except Exception as e:
    print(f"⚠ モデル読み込みスキップ: {e}")
    _bundle = None

# ── feature_importance.csv から剪定シミュレーション ────────────────────
_fi8 = pd.read_csv(REPORTS_DIR / 'feature_importance.csv') if (REPORTS_DIR / 'feature_importance.csv').exists() else pd.DataFrame()
_prune8 = pd.read_csv(REPORTS_DIR / 'feature_pruning_report.csv') if (REPORTS_DIR / 'feature_pruning_report.csv').exists() else pd.DataFrame()

if not _fi8.empty and 'lgb_gain' in _fi8.columns:
    # lgb_gain 順にランキング
    _ranked = _fi8.sort_values('lgb_gain', ascending=False).reset_index(drop=True)
    _ranked['rank'] = _ranked.index + 1
    _total = len(_ranked)
    print(f"総特徴量数: {_total}")

    # 既存の剪定シミュレーション結果を表示
    if not _prune8.empty:
        print("\n=== 既存の剪定シミュレーション結果 ===")
        print(_prune8.to_string(index=False))

    # ── 新規: Top N でのカバレッジ分析 ────────────────────────────────
    _thresholds = [150, 120, 100, 80, 60]
    _sim_results = []
    for n in _thresholds:
        if n > _total:
            n = _total
        _top_n = _ranked.head(n)
        _gain_coverage = _top_n['lgb_gain'].sum() / _ranked['lgb_gain'].sum() * 100
        _shap_coverage = (_top_n['shap_mean'].sum() / _ranked['shap_mean'].sum() * 100
                          if 'shap_mean' in _ranked.columns else np.nan)
        _sim_results.append({
            'n_features': n,
            'gain_coverage_%': round(_gain_coverage, 1),
            'shap_coverage_%': round(_shap_coverage, 1) if not np.isnan(_shap_coverage) else 'N/A',
            'removed': _total - n,
            'removed_gain_%': round(100 - _gain_coverage, 2),
        })

    _sim_df = pd.DataFrame(_sim_results)
    print("\n=== Top N 特徴量カバレッジシミュレーション ===")
    print(_sim_df.to_string(index=False))

    # ── 推奨特徴量数の判定 ──────────────────────────────────────────────
    # gain_coverage >= 98% を維持できる最小特徴量数を推奨とする
    _rec = next((r['n_features'] for r in _sim_results if r['gain_coverage_%'] >= 98.0), _total)
    print(f"\n推奨特徴量数（Gain 98%カバレッジ維持）: {_rec} 列")

    # ── 保存 ────────────────────────────────────────────────────────────
    _sim_path = REPORTS_DIR / 'feature_pruning_sim_ex.csv'
    _sim_df.to_csv(_sim_path, index=False, encoding='utf-8-sig')
    print(f"✓ 保存: {_sim_path}")

    # ── 削除してもよい特徴量リスト（Bottom N-_rec 件）──────────────────
    _removable = _ranked.tail(_total - _rec)[['rank', 'feature', 'lgb_gain', 'shap_mean']].copy()
    print(f"\n削除候補 {len(_removable)} 件（Gain下位）:")
    print(_removable.head(20).to_string(index=False))
    if len(_removable) > 20:
        print(f"  ... 他 {len(_removable) - 20} 件")

    from IPython.display import display
    display(_sim_df)
else:
    print("⚠ feature_importance.csv が見つかりません。04_feature_analysis.ipynb を先に実行してください。")


⚠ モデルファイルが見つかりません。feature_pruning_report.csv の剪定シミュレーション結果を使用します。
総特徴量数: 146

=== 既存の剪定シミュレーション結果 ===
 n_dropped  n_features  cv_mean cv_metric
         0         105  -0.9988     -rmse
        10          95  -0.9988     -rmse
        20          85  -0.9988     -rmse
        30          75  -0.9988     -rmse
        50          55  -0.9988     -rmse
        70          35  -0.9988     -rmse

=== Top N 特徴量カバレッジシミュレーション ===
 n_features  gain_coverage_%  shap_coverage_%  removed  removed_gain_%
        146            100.0            100.0        0            0.00
        120             99.9             99.8       26            0.08
        100             98.8             98.4       46            1.22
         80             95.8             94.7       66            4.20
         60             91.0             88.4       86            8.97

推奨特徴量数（Gain 98%カバレッジ維持）: 146 列
✓ 保存: C:\Users\yuki2\Documents\ws\keiba-ai-pro\notebooks\reports\feature_pruning_sim_ex.csv

削除候補 0 件（Gain下位）:
Emp

,n_features,gain_coverage_%,shap_coverage_%,removed,removed_gain_%
0,146,100.0,100.0,0,0.00
1,120,99.9,99.8,26,0.08
2,100,98.8,98.4,46,1.22
3,80,95.8,94.7,66,4.20
4,60,91.0,88.4,86,8.97
